## Materials Project Database Code 
### This code builds the Materials Project Database

In [ ]:
#Materials Proejct Elasticity and dielectric data, alongside llatice of the materials
from pymatgen.ext.matproj import MPRester
import pandas as pd
from itertools import combinations
import time
import matplotlib.pyplot as plt
import json
import os

# API key and list of elements
api_key = "oda0Y1wP61cFXKCzQGlggpzYUsYxYg9K"
elemental_lst = ["B", "Al", "Ga", "In", "Tl", "Pb", "Sn", "Ge", "Si", "C", "N", "P", "As", "Sb", "Bi", "Po", "Te", "Se", "S", "O"]

# Maximum number of elements in a combination
max_elements_in_combination = 3

# Generate all combinations of elements
element_combinations = []
for r in range(1, max_elements_in_combination + 1):
    element_combinations.extend(combinations(elemental_lst, r))

# Absolute file paths for saving JSON and CSV
directory = "C:/Users/monia/Downloads/MaterialsProj_data/"

# Ensure directory exists, if not create it
os.makedirs(directory, exist_ok=True)

json_file = os.path.join(directory, "filtered_materials_with_combinations.json")
performance_file = os.path.join(directory, "performance_data.json")
elasticity_data_file = os.path.join(directory, "elasticity_data.json")
dielectric_data_file = os.path.join(directory, "dielectric_data.json")
materials_file = os.path.join(directory, "filtered_materials_with_combinations.csv")  # Define materials file

# Helper function to load JSON safely
def load_json_file(file_path):
    if os.path.exists(file_path):
        try:
            with open(file_path, "r") as f:
                return json.load(f)
        except json.JSONDecodeError:
            print(f"Warning: {file_path} is empty or invalid. Starting fresh.")
            return []
    return []

# Initialize storage
all_materials_data = load_json_file(json_file)
performance_data = load_json_file(performance_file)

# Initialize storage for elasticity and dielectric data
elasticity_data = {}
dielectric_data = {}

# Track already processed combinations to avoid duplicates
processed_combinations = {tuple(entry["elements"].split(",")) for entry in all_materials_data}

# Setup live plot
plt.figure(figsize=(10, 6))
plt.title("Performance of Queries by Element Combinations")
plt.xlabel("Combination Index")
plt.ylabel("Query Time (seconds)")
plt.grid()

with MPRester(api_key) as mpr:
    for combo in element_combinations:
        if tuple(combo) in processed_combinations:
            continue  # Skip already processed combinations

        start_time = time.time()  # Start timing the query

        # Query materials summary for specific criteria
        docs = mpr.summary.search(
            elements=list(combo),  # Pass the combination as a list
            band_gap=(0, 0.3),
            energy_above_hull=(0, 0.1),
            fields=["material_id", "formula_pretty", "band_gap", "energy_above_hull"]
        )

        # Process the results
        for doc in docs:
            material_data = {
                "material_id": doc.material_id,
                "formula_pretty": doc.formula_pretty,
                "band_gap": doc.band_gap,
                "energy_above_hull": doc.energy_above_hull,
                "elements": ",".join(combo),  # Join elements as a single string
            }

            # Try retrieving elasticity and dielectric data
            elasticity_data_for_material = mpr.elasticity.get_data_by_id(doc.material_id)
            dielectric_data_for_material = mpr.dielectric.get_data_by_id(doc.material_id)
            material_data["elasticity"] = elasticity_data_for_material
            material_data["dielectric"] = dielectric_data_for_material

            elasticity_data[doc.material_id] = elasticity_data_for_material or "No elasticity data found."
            dielectric_data[doc.material_id] = dielectric_data_for_material or "No dielectric data found."

            all_materials_data.append(material_data)

        # Record time taken for the query
        elapsed_time = time.time() - start_time
        performance_data.append({"combination": ",".join(combo), "time": elapsed_time})

        # Update processed combinations
        processed_combinations.add(tuple(combo))

        # Save and print materials data
        materials_df = pd.DataFrame(all_materials_data)
        materials_df.to_csv(materials_file, index=False)
        print(f"Materials Data (saved to {materials_file}):")
        print(materials_df)

        # Save and print performance data
        performance_df = pd.DataFrame(performance_data)
        performance_df.to_csv(performance_file, index=False)
        print(f"Performance Data (saved to {performance_file}):")
        print(performance_df)

        # Save and print elasticity and dielectric data
        elasticity_df = pd.DataFrame.from_dict(elasticity_data, orient="index").reset_index().rename(columns={"index": "material_id"})
        dielectric_df = pd.DataFrame.from_dict(dielectric_data, orient="index").reset_index().rename(columns={"index": "material_id"})
        elasticity_df.to_csv(elasticity_data_file, index=False)
        dielectric_df.to_csv(dielectric_data_file, index=False)
        print(f"Elasticity Data (saved to {elasticity_data_file}):")
        print(elasticity_df)
        print(f"Dielectric Data (saved to {dielectric_data_file}):")
        print(dielectric_df)

        # Update live plot
        plt.plot(range(len(performance_data)), [p["time"] for p in performance_data], marker="o", linestyle="-", color="b")
        plt.pause(0.1)  # Pause for 0.1 seconds to allow the plot to update
        plt.draw()  # Redraw the plot

# Save the final results and plot
plt.savefig(os.path.join(directory, "performance_plot.png"))
print("Performance plot saved as performance_plot.png")
plt.show()


## Neural Network to Predict Band gap

In [1]:
# Pol Benítez Colominas, March 2024 - May 2024
# Universitat Politècnica de Catalunya

# Crystal Graph Neural Network (CGNN) model for band gap prediction

import csv

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error, mean_absolute_error

import torch
import torch.nn.functional as F
from torch.nn import Linear
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GraphConv
from torch_geometric.nn import global_mean_pool


# Check if a GPU is available
if torch.cuda.is_available():
    device = torch.device('cuda')
    print("GPU is available. Using GPU.")
else:
    device = torch.device('cpu')
    print("GPU not available. Using CPU.")


# Define the machine learning model parameters
num_epochs = 100
batch_size = 256
learning_rate = 0.00001
dropout = 0.5


# Load the normalized graphs and save them in an array
file_list = []

with open('graphs-bg.csv', 'r') as csv_file:
    csv_reader = csv.reader(csv_file)
    next(csv_reader)
    for row in csv_reader:
        file_list.append(row[0])

dataset_graphs = []

path_data = 'normalized_graphs/'

df_materials = pd.read_csv('graphs-bg.csv')

for filename in file_list:
    data = torch.load(path_data + filename + '.pt')
    dataset_graphs.append(data)

print(f'A total of {len(dataset_graphs)} graphs loaded.')


# Define the size of the train and test set
train_size = int(0.9 * len(dataset_graphs))
test_size  = len(dataset_graphs) - train_size

print(f'The train set contains {train_size} graphs')
print(f'The test set contains {test_size} graphs')


# Create the train and test sets
train_dataset, test_dataset = torch.utils.data.random_split(dataset_graphs, [train_size, test_size])


# Generate the train and test loaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=len(test_dataset), shuffle=False)


# Define the CGNN model
class GCNN(torch.nn.Module):
    """
    Graph Convolution Neural Network model
    """

    def __init__(self, features_channels, hidden_channels):
        super(GCNN, self).__init__()
        torch.manual_seed(12345)

        # Convolution layers
        self.conv1 = GraphConv(features_channels, hidden_channels)
        self.conv2 = GraphConv(hidden_channels, hidden_channels)

        # Linear layers
        self.lin1 = Linear(hidden_channels, 16)
        self.lin2 = Linear(16, 1)

    def forward(self, x, edge_index, edge_attr, batch):

        # Node embedding
        x = self.conv1(x, edge_index, edge_attr)
        x = x.relu()
        x = self.conv2(x, edge_index, edge_attr)

        # Mean pooling to reduce dimensionality
        x = global_mean_pool(x, batch)  

        # Apply neural network for regression prediction problem

        x = F.dropout(x, p=0.5, training=self.training)
        x = self.lin1(x)
        x = x.relu()
        x = self.lin2(x)
        x = x.relu()

        return x
    
model = GCNN(features_channels=dataset_graphs[0].num_node_features, hidden_channels=batch_size)
print(model)


# Define the optimizer and criterion
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = torch.nn.MSELoss()


# Define functions to train the model with the train set and evaluate its performance over the train and test sets
def train(model, criterion, train_loader, optimizer):
    """
    Train the model with the train set
    """

    model.train()
    total_loss = 0
    all_predictions = []
    all_ground_truths = []

    for data in train_loader:  # Iterate in batches over the training dataset
        # Ensure tensors are on the right device and dtype
        data.x = data.x.to(device).float()
        data.edge_index = data.edge_index.to(device).long()
        data.edge_attr = data.edge_attr.to(device).float()
        data.y = data.y.to(device).float()
        
        
        out = model(data.x, data.edge_index, data.edge_attr, data.batch).to(device).squeeze(-1)
        loss = criterion(out, data.y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item()
        all_predictions.append(out.detach().cpu())
        all_ground_truths.append(data.y.detach().cpu())
    
    average_loss = total_loss / len(train_loader)

    return average_loss, all_predictions, all_ground_truths


def test(model, criterion, test_loader):
    """
    Check the performance of the model over the test set
    """

    model.eval()
    total_loss = 0
    all_predictions = []
    all_ground_truths = []
    with torch.no_grad():
        for data in test_loader:  # Iterate in batches over the test dataset
            # Ensure tensors are on the right device and dtype
            data.x = data.x.to(device).float()
            data.edge_index = data.edge_index.to(device).long()
            data.edge_attr = data.edge_attr.to(device).float()
            data.y = data.y.to(device).float()

            out = model(data.x, data.edge_index, data.edge_attr, data.batch).to(device).squeeze(-1)
            loss = criterion(out, data.y)

            total_loss += loss.item()
            all_predictions.append(out.cpu())
            all_ground_truths.append(data.y.cpu())

    average_loss = total_loss / len(test_loader)

    return average_loss, all_predictions, all_ground_truths


# Loop over the epochs to train the model
train_losses = []
test_losses = []

for epoch in range(num_epochs):
    train_loss, train_predictions, train_ground_truths = train(model, criterion, train_loader, optimizer)
    test_loss, test_predictions, test_ground_truths = test(model, criterion, test_loader)

    # Append losses
    train_losses.append(train_loss)
    test_losses.append(test_loss)

    print(f'Epoch {epoch + 1} of a total of {num_epochs}')
    print(f'     Train loss:   {train_loss}')
    print(f'     Test loss:    {test_loss}')


# Plot the train/test loss with epochs
plt.figure()
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.plot(np.linspace(1, num_epochs, num_epochs), train_losses, label='train')
plt.plot(np.linspace(1, num_epochs, num_epochs), test_losses, label='test')
plt.legend()
plt.tight_layout()
plt.savefig('loss_plot.pdf')


# Show the predicted value vs DFT value
real_value_train = []
predicted_value_train = []
real_value_test = []
predicted_value_test = []

model.eval()

for num_graph in range(len(train_dataset)):
    graph = train_dataset[num_graph]

    real_value_train.append(graph.y[0])

    graph.x = graph.x.to(device).float()
    graph.edge_index = graph.edge_index.to(device).long()
    graph.edge_attr = graph.edge_attr.to(device).float()
    graph.y = graph.y.to(device).float()
    graph = graph.to(device)

    model = model.to(device).float()

    # Make prediction
    with torch.no_grad():  # Disable gradient calculation for inference
        prediction = model(graph.x, graph.edge_index, graph.edge_attr, graph.batch).to(device)

    # Convert prediction to CPU if necessary
    prediction = prediction.cpu()
    predicted_value_train.append(prediction[0][0])


for num_graph in range(len(test_dataset)):
    graph = test_dataset[num_graph]

    real_value_test.append(graph.y[0])

    graph.x = graph.x.to(device).float()
    graph.edge_index = graph.edge_index.to(device).long()
    graph.edge_attr = graph.edge_attr.to(device).float()
    graph.y = graph.y.to(device).float()
    graph = graph.to(device)

    batch = torch.zeros(graph.num_nodes, dtype=torch.long).to(device)

    model = model.to(device).float()

    # Make prediction
    with torch.no_grad():  # Disable gradient calculation for inference
        prediction = model(graph.x, graph.edge_index, graph.edge_attr, graph.batch).to(device)

    # Convert prediction to CPU if necessary
    prediction = prediction.cpu()
    predicted_value_test.append(prediction[0][0])

plt.figure()
plt.xlabel('DFT computed band gap (eV)')
plt.ylabel('Predicted band gap (eV)')
max_value = max(np.max(real_value_train), np.max(real_value_test), np.max(predicted_value_train), np.max(predicted_value_test))
plt.xlim(0, max_value)
plt.ylim(0, max_value)
plt.plot(real_value_train[:], predicted_value_train[:], linestyle='', marker='o', alpha=0.6, color='lightsteelblue', label='train')
plt.plot(real_value_test[:], predicted_value_test[:], linestyle='', marker='o', alpha=0.6, color='salmon', label='test')
plt.plot([0,max_value], [0,max_value], linestyle='--', color='royalblue')
plt.legend()
plt.tight_layout()
plt.savefig('predictions_plot.pdf')


# Save some metrics of the model
mse_train = mean_squared_error(real_value_train,predicted_value_train)
mse_test = mean_squared_error(real_value_test,predicted_value_test)

mae_train = mean_absolute_error(real_value_train,predicted_value_train)
mae_test = mean_absolute_error(real_value_test,predicted_value_test)

metrics_file = open('metrics.txt', 'w')
metrics_file.write('Mean Squared Error (MSE) and Mean Absolute Error (MAE) metrics for train and test set:\n')
metrics_file.write(f'MSE train:   {mse_train}\n')
metrics_file.write(f'MSE test:    {mse_test}\n')
metrics_file.write(f'MAE train:   {mae_train}\n')
metrics_file.write(f'MAE test:    {mae_test}\n')
metrics_file.write('\n')
metrics_file.write('\n')
metrics_file.write('Train and test loss after each epoch:\n')
for epoch in range(num_epochs):
    metrics_file.write(f'Epoch {epoch + 1} of a total of {num_epochs}\n')
    metrics_file.write(f'     Train loss:   {train_losses[epoch]}\n')
    metrics_file.write(f'     Test loss:    {test_losses[epoch]}\n')
metrics_file.close()


# Save the model
model_path = 'trained_model'
torch.save(model.state_dict(), model_path)


# Open a model
#model = GCNN(features_channels=dataset_graphs[0].num_node_features, hidden_channels=batch_size)
#model.load_state_dict(torch.load(model_path))
#model.eval()

GPU not available. Using CPU.


FileNotFoundError: [Errno 2] No such file or directory: 'graphs-bg.csv'